# Step 3: Data Cleaning & Preprocessing
**Project:** Taobao CTR Prediction — Big Data Pipeline  
**Target:** `clk` (1 = clicked, 0 = not clicked)  
**Steps:** Missing Values → Duplicates → Join Tables → Feature Engineering → Encoding → Scaling → Save

## Cell 0 — HADOOP Setup 

In [1]:
import os

os.environ["HADOOP_HOME"]     = "D:\\Apps\\hadoop"
os.environ["hadoop.home.dir"] = "D:\\Apps\\hadoop"
os.environ["PATH"]            = "D:\\Apps\\hadoop\\bin;" + os.environ["PATH"]

print(" HADOOP_HOME:", os.environ["HADOOP_HOME"])
print(" Hadoop configured!")

 HADOOP_HOME: D:\Apps\hadoop
 Hadoop configured!


## Cell 1 — Spark Session

In [2]:
import findspark
findspark.init()

import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, isnull, count, mean, hour,
    from_unixtime, dayofweek, sum as spark_sum,
    round as spark_round
)
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
import pandas as pd

DATA_PATH   = r"C:\Users\Ahmed\Desktop\Project Big-Data\ProjectBigData\archive"
OUTPUT_PATH = r"C:\Users\Ahmed\Desktop\Project Big-Data\ProjectBigData\processed_data"

spark = SparkSession.builder \
    .appName("Taobao_CTR_Preprocessing") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(" Spark Version:", spark.version)
print(" Spark Session Started!")

 Spark Version: 4.1.1
 Spark Session Started!


## Cell 2 — Load Data

In [3]:
print(" Loading datasets...\n")

raw_sample_df = spark.read.csv(
    os.path.join(DATA_PATH, "raw_sample.csv"),
    header=True, inferSchema=True
)
ad_feature_df = spark.read.csv(
    os.path.join(DATA_PATH, "ad_feature.csv"),
    header=True, inferSchema=True
)
user_profile_df = spark.read.csv(
    os.path.join(DATA_PATH, "user_profile.csv"),
    header=True, inferSchema=True
)

print(f" raw_sample   : {raw_sample_df.count():,} records")
print(f" ad_feature   : {ad_feature_df.count():,} records")
print(f" user_profile : {user_profile_df.count():,} records")

 Loading datasets...

 raw_sample   : 26,557,961 records
 ad_feature   : 846,811 records
 user_profile : 1,048,575 records


## Cell 3 — Handle Missing Values

In [4]:
print(" STEP 1: HANDLING MISSING VALUES")
print("=" * 50)

ad_feature_clean = ad_feature_df.fillna({"brand": "0"})


user_profile_clean = user_profile_df.fillna({
    "pvalue_level"          : 0,
    "new_user_class_level" : 0,
    "cms_segid"             : 0
})

def check_nulls(df, name):
    null_expr = [spark_sum(when(isnull(c), 1).otherwise(0)).alias(c) for c in df.columns]
    result    = df.select(null_expr).collect()[0].asDict()
    total     = sum(result.values())
    print(f"  {name}: {total} nulls remaining")

check_nulls(ad_feature_clean,   "ad_feature")
check_nulls(user_profile_clean, "user_profile")
print("\n Missing values handled!")

 STEP 1: HANDLING MISSING VALUES
  ad_feature: 0 nulls remaining
  user_profile: 0 nulls remaining

 Missing values handled!


## Cell 4 — Remove Duplicates

In [5]:
print(" STEP 2: REMOVING DUPLICATES")
print("=" * 50)

before           = raw_sample_df.count()
raw_sample_clean = raw_sample_df.dropDuplicates(["user", "time_stamp", "adgroup_id"])
after            = raw_sample_clean.count()

print(f"  raw_sample Before : {before:,}")
print(f"  raw_sample After  : {after:,}")
print(f"  Duplicates Removed: {before - after:,}")

ad_feature_clean = ad_feature_clean.dropDuplicates(["adgroup_id"])
print(f"\n  ad_feature unique ads: {ad_feature_clean.count():,}")
print("\n Duplicates removed!")

 STEP 2: REMOVING DUPLICATES
  raw_sample Before : 26,557,961
  raw_sample After  : 26,557,961
  Duplicates Removed: 0

  ad_feature unique ads: 846,811

 Duplicates removed!


## Cell 5 — Join Tables

In [6]:
print(" STEP 3: JOINING TABLES")
print("=" * 50)

df = raw_sample_clean.join(ad_feature_clean, on="adgroup_id", how="left")
print(f"  After join with ad_feature   : {df.count():,} records")

df = df.join(
    user_profile_clean,
    df["user"] == user_profile_clean["userid"],
    how="left"
).drop("userid")

print(f"  After join with user_profile : {df.count():,} records")
print(f"  Total Columns                : {len(df.columns)}")
print("\n All tables joined!")

 STEP 3: JOINING TABLES
  After join with ad_feature   : 26,557,961 records
  After join with user_profile : 26,557,961 records
  Total Columns                : 19

 All tables joined!


## Cell 6 — Feature Engineering

In [ ]:
print(" STEP 4: FEATURE ENGINEERING")
print("=" * 50)

df = df.withColumn("hour",        hour(from_unixtime(col("time_stamp"))))
df = df.withColumn("day_of_week", dayofweek(from_unixtime(col("time_stamp"))))
df = df.withColumn("is_weekend",  when(col("day_of_week").isin([1, 7]), 1).otherwise(0))
df = df.withColumn("time_segment",
    when(col("hour").between(0, 6),   0)
    .when(col("hour").between(7, 11),  1)
    .when(col("hour").between(12, 17), 2)
    .otherwise(3)
)

ad_ctr = df.groupBy("adgroup_id").agg(
    spark_round(mean("clk"), 4).alias("ad_historical_ctr"),
    count("*").alias("ad_impression_count")
)
df = df.join(ad_ctr, on="adgroup_id", how="left")

user_activity = df.groupBy("user").agg(
    spark_sum("clk").alias("user_total_clicks"),
    count("*").alias("user_total_impressions")
)
user_activity = user_activity.withColumn(
    "user_ctr",
    spark_round(col("user_total_clicks") / col("user_total_impressions"), 4)
)
df = df.join(user_activity, on="user", how="left")

df = df.fillna({
    "ad_historical_ctr"     : 0.0,
    "ad_impression_count"   : 0,
    "user_total_clicks"     : 0,
    "user_total_impressions": 0,
    "user_ctr"              : 0.0,
    "price"                 : 0.0,
    "cms_segid"             : 0,
    "cms_group_id"          : 0,
    "final_gender_code"     : 0,
    "age_level"             : 0,
    "pvalue_level"          : 0,
    "shopping_level"        : 0,
    "occupation"            : 0
})

print("   hour, day_of_week, is_weekend, time_segment")
print("   ad_historical_ctr, ad_impression_count")
print("   user_total_clicks, user_ctr")
print(f"\n  Total Columns Now: {len(df.columns)}")
print("\n Feature engineering done!")

 STEP 4: FEATURE ENGINEERING
   hour, day_of_week, is_weekend, time_segment
   ad_historical_ctr, ad_impression_count
   user_total_clicks, user_ctr

  Total Columns Now: 28

 Feature engineering done!


In [8]:
df = df.drop('nonclk')
df.printSchema()

root
 |-- user: integer (nullable = true)
 |-- adgroup_id: integer (nullable = true)
 |-- time_stamp: integer (nullable = true)
 |-- pid: string (nullable = true)
 |-- clk: integer (nullable = true)
 |-- cate_id: integer (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- customer: integer (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = false)
 |-- cms_segid: integer (nullable = false)
 |-- cms_group_id: integer (nullable = false)
 |-- final_gender_code: integer (nullable = false)
 |-- age_level: integer (nullable = false)
 |-- pvalue_level: integer (nullable = false)
 |-- shopping_level: integer (nullable = false)
 |-- occupation: integer (nullable = false)
 |-- new_user_class_level: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- time_segment: integer (nullable = false)
 |-- ad_historical_ctr: double (nullable = false)
 |-- a

In [9]:
df.head()

Row(user=12, adgroup_id=411858, time_stamp=1494302581, pid='430548_1007', clk=0, cate_id=4283, campaign_id=49595, customer=29158, brand='274972', price=258.0, cms_segid=67, cms_group_id=9, final_gender_code=1, age_level=3, pvalue_level=2, shopping_level=3, occupation=0, new_user_class_level=3, hour=6, day_of_week=3, is_weekend=0, time_segment=0, ad_historical_ctr=0.0855, ad_impression_count=152, user_total_clicks=0, user_total_impressions=3, user_ctr=0.0)

## Cell 7 — Encoding + Assembly + Scaling

In [10]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

print("STEP 5: THE 'HACK' PIPELINE (EVERYTHING AS CONTINUOUS)")
print("=" * 50)

stages = []

string_cols  = [c for c, t in df.dtypes if t == "string"]
encoded_map  = {}

for s_col in string_cols:
    out_col = s_col + "_encoded"
    indexer = StringIndexer(inputCol=s_col, outputCol=out_col, handleInvalid="keep")
    stages.append(indexer)
    encoded_map[s_col] = out_col

actual_num_cols = [
    "price", "ad_historical_ctr", "ad_impression_count",
    "user_total_clicks", "user_ctr"
]

integer_id_cols = [
    "adgroup_id", "cate_id", "campaign_id", "cms_segid", "cms_group_id", 
    "final_gender_code", "age_level", "pvalue_level", "shopping_level",
    "occupation", "hour", "day_of_week", "is_weekend", "time_segment"
]

string_encoded_cols = [encoded_map[c] for c in string_cols if c in df.columns]

all_numeric_features = actual_num_cols + integer_id_cols + string_encoded_cols

assembler = VectorAssembler(inputCols=all_numeric_features, outputCol="features_raw", handleInvalid="keep")
stages.append(assembler)

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
stages.append(scaler)

pipeline       = Pipeline(stages=stages)
pipeline_model = pipeline.fit(df)
df_final       = pipeline_model.transform(df)

cols_to_keep = ["features", "clk"]
if "class_weight" in df_final.columns:
    cols_to_keep.append("class_weight")
    
df_final = df_final.select(*cols_to_keep)

print(f"df_final rows: {df_final.count():,}")
print("Hack Applied! Random Forest will treat ALL features as continuous numbers. No maxBins error!")

STEP 5: THE 'HACK' PIPELINE (EVERYTHING AS CONTINUOUS)
df_final rows: 26,557,961
Hack Applied! Random Forest will treat ALL features as continuous numbers. No maxBins error!


## Cell 8 — Class Imbalance

In [11]:
print(" STEP 6: HANDLE CLASS IMBALANCE")
print("=" * 50)

total     = df_final.count()
clicks    = df_final.filter(col("clk") == 1).count()
no_clicks = total - clicks

weight_for_click    = total / (2 * clicks)
weight_for_no_click = total / (2 * no_clicks)

df_final = df_final.withColumn(
    "class_weight",
    when(col("clk") == 1, weight_for_click).otherwise(weight_for_no_click)
)

print(f"  Clicks    : {clicks:,}  ({clicks/total*100:.2f}%)")
print(f"  No Click  : {no_clicks:,} ({no_clicks/total*100:.2f}%)")
print(f"  Weight(1) : {weight_for_click:.2f}")
print(f"  Weight(0) : {weight_for_no_click:.2f}")
print("\n Class weights added!")

 STEP 6: HANDLE CLASS IMBALANCE
  Clicks    : 1,366,056  (5.14%)
  No Click  : 25,191,905 (94.86%)
  Weight(1) : 9.72
  Weight(0) : 0.53

 Class weights added!


## Cell 9 — Train/Test Split

In [12]:
print(" STEP 7: TRAIN / TEST SPLIT")
print("=" * 50)

TEST_START_TIMESTAMP = 1494604800

train_df = df_final.filter(col("time_stamp") <  TEST_START_TIMESTAMP)
test_df  = df_final.filter(col("time_stamp") >= TEST_START_TIMESTAMP)

train_count = train_df.count()
test_count  = test_df.count()
total_count = train_count + test_count

print(f"  Train (days 1-7) : {train_count:,} ({train_count/total_count*100:.1f}%)")
print(f"  Test  (day 8)    : {test_count:,}  ({test_count/total_count*100:.1f}%)")
print("\n Train/Test split done!")

 STEP 7: TRAIN / TEST SPLIT
  Train (days 1-7) : 23,249,296 (87.5%)
  Test  (day 8)    : 3,308,665  (12.5%)

 Train/Test split done!


## Cell 10 — Save Data

In [13]:
print(" STEP 8: SAVE PROCESSED DATA")
print("=" * 50)

final_cols    = ["features", "clk", "class_weight"]
TRAIN_PATH    = OUTPUT_PATH + "processed_data/train"
TEST_PATH     = OUTPUT_PATH + "processed_data/test"
PIPELINE_PATH = OUTPUT_PATH + "pipeline_model"

print(" Saving train data...")
train_df.select(final_cols).write.mode("overwrite").parquet(TRAIN_PATH)
print(f"   Train saved → {TRAIN_PATH}")

print(" Saving test data...")
test_df.select(final_cols).write.mode("overwrite").parquet(TEST_PATH)
print(f"   Test  saved → {TEST_PATH}")

print(" Saving pipeline...")
pipeline_model.write().overwrite().save(PIPELINE_PATH)
print(f"   Pipeline saved → {PIPELINE_PATH}")

print("\n All data saved!")

 STEP 8: SAVE PROCESSED DATA
 Saving train data...
   Train saved → C:\Users\Ahmed\Desktop\Project Big-Data\ProjectBigData\processed_dataprocessed_data/train
 Saving test data...
   Test  saved → C:\Users\Ahmed\Desktop\Project Big-Data\ProjectBigData\processed_dataprocessed_data/test
 Saving pipeline...
   Pipeline saved → C:\Users\Ahmed\Desktop\Project Big-Data\ProjectBigData\processed_datapipeline_model

 All data saved!


In [15]:
import warnings
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, LongType

warnings.filterwarnings('ignore')

print("Starting PySpark Session...")
spark = SparkSession.builder \
    .appName("Taobao_CTR_Inference") \
    .master("local[*]") \
    .config("spark.driver.extraJavaOptions", "--add-exports java.base/sun.nio.ch=ALL-UNNAMED") \
    .config("spark.executor.extraJavaOptions", "--add-exports java.base/sun.nio.ch=ALL-UNNAMED") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

pipeline_path = r"C:\Users\Ahmed\Desktop\Project Big-Data\ProjectBigData\processed_datapipeline_model"
print(f"Loading Pipeline from: {pipeline_path}")
pipeline_model = PipelineModel.load(pipeline_path)

schema = StructType([
    StructField("user", IntegerType(), True),
    StructField("adgroup_id", IntegerType(), True),
    StructField("time_stamp", IntegerType(), True),
    StructField("pid", StringType(), True),
    StructField("clk", IntegerType(), True),
    StructField("cate_id", IntegerType(), True),
    StructField("campaign_id", IntegerType(), True),
    StructField("customer", IntegerType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("cms_segid", IntegerType(), True),
    StructField("cms_group_id", IntegerType(), True),
    StructField("final_gender_code", IntegerType(), True),
    StructField("age_level", IntegerType(), True),
    StructField("pvalue_level", IntegerType(), True),
    StructField("shopping_level", IntegerType(), True),
    StructField("occupation", IntegerType(), True),
    StructField("new_user_class_level", IntegerType(), True),
    StructField("hour", IntegerType(), True),
    StructField("day_of_week", IntegerType(), True),
    StructField("is_weekend", IntegerType(), True),
    StructField("time_segment", IntegerType(), True),
    StructField("ad_historical_ctr", DoubleType(), True),
    StructField("ad_impression_count", LongType(), True),
    StructField("user_total_clicks", LongType(), True),
    StructField("user_total_impressions", LongType(), True),
    StructField("user_ctr", DoubleType(), True)
])

data = [(
    123456,
    1001,
    1494032110,
    "430548_1007",
    0,
    7266,
    5001,
    75195,
    "311200",
    198.0,
    56,
    8,
    1,
    1,
    2,
    1,
    0,
    1,
    12,
    7,
    0,
    2,
    0.05,
    1500,
    10,
    100,
    0.1
)]

input_df = spark.createDataFrame(data, schema=schema)

print("\nSchema of Input:")
input_df.printSchema()

print("Transforming Data through Pipeline...")
transformed_df = pipeline_model.transform(input_df)

print("\nTransformed Output (Features Ready for Model Prediction):")
transformed_df.select("features").show(truncate=False)

Starting PySpark Session...
Loading Pipeline from: C:\Users\Ahmed\Desktop\Project Big-Data\ProjectBigData\processed_datapipeline_model

Schema of Input:
root
 |-- user: integer (nullable = true)
 |-- adgroup_id: integer (nullable = true)
 |-- time_stamp: integer (nullable = true)
 |-- pid: string (nullable = true)
 |-- clk: integer (nullable = true)
 |-- cate_id: integer (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- customer: integer (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cms_segid: integer (nullable = true)
 |-- cms_group_id: integer (nullable = true)
 |-- final_gender_code: integer (nullable = true)
 |-- age_level: integer (nullable = true)
 |-- pvalue_level: integer (nullable = true)
 |-- shopping_level: integer (nullable = true)
 |-- occupation: integer (nullable = true)
 |-- new_user_class_level: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 